# Test Perception — Ball, Obstacle, and Basket Detection

**Purpose:** Test and tune all perception modules with live camera feed.

**Use this to:**
- Verify ball detection for all colors
- Test obstacle/boundary detection
- Validate basket detection
- Tune HSV ranges on-site

## 1. Setup

In [ ]:
import sys
import os
import time
import yaml
import cv2
import numpy as np
from IPython.display import display, Image
import ipywidgets.widgets as widgets

project_root = '/home/steve/Notebooks/Projects/AI & Robotics Hackathon Berlin/ITQ_HackLab_Team_2'
if project_root not in sys.path:
    sys.path.insert(0, project_root)

from src.perception.ball_detector import BallDetector
from src.perception.obstacle_detector import ObstacleDetector
from src.perception.basket_detector import BasketDetector
from src.hardware.camera import CameraController

print("✓ Modules imported")

## 2. Load Config & Initialize

In [ ]:
config_path = os.path.join(project_root, 'config.yaml')
with open(config_path, 'r') as f:
    config = yaml.safe_load(f)

# Initialize camera
camera_ctrl = CameraController(config)
if not camera_ctrl.initialize():
    raise Exception("Camera failed!")
camera_ctrl.center()
camera_ctrl.look_down()

# Initialize detectors
ball_detector = BallDetector(config)
obstacle_detector = ObstacleDetector(config)
basket_detector = BasketDetector(config)

print("✓ All initialized")

## 3. Live Detection Test

In [ ]:
# Capture and process frame
frame = camera_ctrl.read()

if frame is not None:
    # Run all detectors
    balls = ball_detector.detect(frame)
    obstacle = obstacle_detector.detect_combined(frame)
    basket = basket_detector.detect(frame)
    
    # Print results
    print("="*50)
    print("DETECTION RESULTS")
    print("="*50)
    
    print(f"\nBalls detected: {len(balls)}")
    for ball in balls:
        color, centroid, distance, area = ball
        print(f"  - {color.upper()}: {distance:.0f} cm at {centroid}")
    
    print(f"\nBoundary: {'YES' if obstacle['boundary_detected'] else 'NO'}")
    if obstacle['boundary_detected']:
        print(f"  Yellow pixels: {obstacle['yellow_pixels']}")
        print(f"  Turn: {obstacle['turn_direction']}")
    
    print(f"\nObstacle: {'YES' if obstacle['obstacle_detected'] else 'NO'}")
    if obstacle['obstacle_detected']:
        print(f"  Edge pixels: {obstacle['edge_pixels']}")
        print(f"  Turn: {obstacle['turn_direction']}")
    
    print(f"\nBasket: {'FOUND' if basket['basket_found'] else 'NOT FOUND'}")
    if basket['basket_found']:
        print(f"  Distance: {basket['distance']:.0f} cm")
        print(f"  Bearing: {basket['bearing']:.0f}°")
    
    # Draw overlays
    overlay = ball_detector.draw_detections(frame, balls)
    overlay = obstacle_detector.draw_detections(overlay, obstacle)
    overlay = basket_detector.draw_detection(overlay, basket)
    
    # Display
    _, buffer = cv2.imencode('.jpg', overlay)
    display(Image(data=buffer.tobytes()))
else:
    print("ERROR: No frame captured")

## 4. Continuous Live Feed (Run to test continuously)

In [ ]:
import ipywidgets as widgets
from IPython.display import display
import threading

# Create image widget
image_widget = widgets.Image(format='jpeg', width=640, height=480)
status_widget = widgets.HTML(value="<b>Status:</b> Ready")

running = False

def update_feed():
    while running:
        frame = camera_ctrl.read()
        if frame is not None:
            # Run detectors
            balls = ball_detector.detect(frame)
            obstacle = obstacle_detector.detect_combined(frame)
            basket = basket_detector.detect(frame)
            
            # Draw overlays
            overlay = ball_detector.draw_detections(frame, balls)
            overlay = obstacle_detector.draw_detections(overlay, obstacle)
            overlay = basket_detector.draw_detection(overlay, basket)
            
            # Update image
            _, buffer = cv2.imencode('.jpg', overlay)
            image_widget.value = buffer.tobytes()
            
            # Update status
            status = f"<b>Balls:</b> {len(balls)} | "
            status += f"<b>Boundary:</b> {'YES' if obstacle['boundary_detected'] else 'NO'} | "
            status += f"<b>Obstacle:</b> {'YES' if obstacle['obstacle_detected'] else 'NO'} | "
            status += f"<b>Basket:</b> {'FOUND' if basket['basket_found'] else 'NO'}"
            status_widget.value = status
        
        time.sleep(0.1)  # 10 Hz

def start_feed(b):
    global running
    running = True
    thread = threading.Thread(target=update_feed)
    thread.start()

def stop_feed(b):
    global running
    running = False

start_button = widgets.Button(description="Start Live Feed", button_style='success')
stop_button = widgets.Button(description="Stop", button_style='danger')

start_button.on_click(start_feed)
stop_button.on_click(stop_feed)

display(widgets.HBox([start_button, stop_button]))
display(status_widget)
display(image_widget)

## 5. HSV Tuning Helper

Use this to find optimal HSV ranges for balls in current lighting.

In [ ]:
# Interactive HSV tuning
frame = camera_ctrl.read()

if frame is not None:
    hsv = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)
    
    # Sample center pixel
    h, w = frame.shape[:2]
    center_hsv = hsv[h//2, w//2]
    
    print("Center pixel HSV:")
    print(f"  H: {center_hsv[0]}")
    print(f"  S: {center_hsv[1]}")
    print(f"  V: {center_hsv[2]}")
    print("\nPlace ball in center of frame and run this cell to get HSV values.")
    print("\nSuggested range (±20 for H, ±50 for S, ±50 for V):")
    print(f"  hsv_lower: [{max(0, center_hsv[0]-20)}, {max(0, center_hsv[1]-50)}, {max(0, center_hsv[2]-50)}]")
    print(f"  hsv_upper: [{min(180, center_hsv[0]+20)}, {min(255, center_hsv[1]+50)}, {min(255, center_hsv[2]+50)}]")

## 6. Cleanup

In [ ]:
running = False
time.sleep(0.5)
camera_ctrl.release()
print("✓ Cleanup complete")